# MethylLlama — Embedding Analysis

Visualise the CLS embeddings extracted from the pretrained encoder using UMAP.
We show two views:
1. **Age gradient** — does the 256-D CLS space encode a continuous age signal?
2. **Tissue structure** — does it separate tissue types without supervision?

These are the same UMAP plots as Figure 3 in the thesis, reproduced on the
120-sample public demo subset.

> **Prerequisite**: run `quickstart.ipynb` first (or execute the cells below
> which re-load the model and extract embeddings).

In [ ]:
import sys
from pathlib import Path

repo_root = Path(".").resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import os
import torch
import anndata as ad
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

## 1. Load model and demo data

In [ ]:
from huggingface_hub import hf_hub_download

from bmfm_methylation.llama.model import MethylLlamaConfig
from bmfm_methylation.llama.wced_llama import WCEDLlamaModule

# Patch torch.load for PyTorch >= 2.6 (weights_only=True breaks Lightning ckpts)
_orig_load = torch.load
def _safe_load(*args, **kwargs):
    kwargs["weights_only"] = False
    return _orig_load(*args, **kwargs)
torch.load = _safe_load

# Download checkpoint
ckpt_path = hf_hub_download(
    repo_id="netanelazran1/MethylLlama",
    filename="pretrain_llama_epoch98_val0.0059.ckpt",
)
ckpt = torch.load(ckpt_path, map_location="cpu")

vocab_size = ckpt["state_dict"]["encoder.embeddings.cpg_sites_embeddings.weight"].shape[0]

model_config = MethylLlamaConfig(
    hidden_size=256, num_hidden_layers=4, num_attention_heads=4,
    intermediate_size=320, vocab_size=vocab_size,
)
module = WCEDLlamaModule(
    model_config=model_config,
    vocab_size=ckpt["hyper_parameters"]["vocab_size"],
)
module.load_state_dict(ckpt["state_dict"])
encoder = module.encoder.to(DEVICE).eval()
print("Model loaded.")

In [ ]:
# Load demo data
adata = ad.read_h5ad("data/methylllama_demo_120samples.h5ad")
cpg_names = list(adata.var_names)
print(f"Demo: {adata.shape[0]} samples × {adata.shape[1]} CpGs")

# Build vocab dict directly from the pretrained 49k-CpG vocabulary
vocab_path = hf_hub_download(
    repo_id="netanelazran1/MethylLlama",
    filename="tokenizer/cpg_sites/vocab.txt",
)
cpg_vocab = {}
with open(vocab_path) as f:
    for idx, line in enumerate(f):
        cpg_vocab[line.strip()] = idx

cls_id = cpg_vocab["[CLS]"]
unk_id = cpg_vocab["[UNK]"]

demo_ids = [cpg_vocab.get(c, unk_id) for c in cpg_names]
n_known  = sum(1 for i in demo_ids if i != unk_id)
print(f"Vocabulary: {len(cpg_vocab)} CpG tokens")
print(f"Demo CpGs in pretrained vocab: {n_known}/{len(cpg_names)} ({100*n_known/len(cpg_names):.1f}%)")

## 2. Extract CLS embeddings for all 120 samples

In [ ]:
def extract_cls(adata, encoder, cpg_vocab, n_cpg=8000, batch_size=16, seed=42):
    """Extract pooler_output (CLS) for all samples using known CpGs only."""
    rng    = np.random.default_rng(seed)
    unk_id = cpg_vocab["[UNK]"]
    cls_id = cpg_vocab["[CLS]"]

    X = adata.X
    if hasattr(X, "toarray"):
        X = X.toarray()
    X = X.astype(np.float32)

    names   = list(adata.var_names)
    all_ids = np.array([cpg_vocab.get(c, unk_id) for c in names])

    # Use only CpGs present in the pretrained vocab
    known_mask = all_ids != unk_id
    known_idx  = np.where(known_mask)[0]
    n_cpg      = min(n_cpg, len(known_idx))

    sel_local     = rng.choice(len(known_idx), size=n_cpg, replace=False)
    sel_idx       = known_idx[sel_local]
    sel_token_ids = all_ids[sel_idx].tolist()
    L             = n_cpg + 1

    all_cls = []
    n = adata.shape[0]

    with torch.no_grad():
        for start in range(0, n, batch_size):
            end   = min(start + batch_size, n)
            betas = X[start:end, sel_idx].copy()
            betas[np.isnan(betas)] = 0.5

            cpg_ids   = torch.tensor([[cls_id] + sel_token_ids] * (end - start),
                                     dtype=torch.float32)
            beta_vals = torch.cat([
                torch.full((end - start, 1), -2.0),
                torch.tensor(betas, dtype=torch.float32)
            ], dim=1)
            input_ids = torch.stack([cpg_ids, beta_vals], dim=1).to(DEVICE)  # [B,2,L]
            mask      = torch.ones(end - start, L, dtype=torch.long).to(DEVICE)

            out = encoder(input_ids=input_ids, attention_mask=mask)
            all_cls.append(out.pooler_output.cpu().numpy())

    return np.vstack(all_cls)


# Use up to 8000 known CpGs for maximum information
cls_emb = extract_cls(adata, encoder, cpg_vocab, n_cpg=8000)
print(f"CLS embeddings: {cls_emb.shape}")

## 3. UMAP

In [ ]:
import umap

reducer = umap.UMAP(n_neighbors=15, min_dist=0.3, random_state=42)
emb_2d  = reducer.fit_transform(cls_emb)
print(f"UMAP output: {emb_2d.shape}")

### 3a. Age gradient

In [ ]:
ages = adata.obs["age"].astype(float).values

fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.scatter(emb_2d[:, 0], emb_2d[:, 1],
                c=ages, cmap="plasma", s=60, alpha=0.85, edgecolors="none")
plt.colorbar(sc, ax=ax, label="Age (years)")
ax.set_xlabel("UMAP 1", fontsize=12)
ax.set_ylabel("UMAP 2", fontsize=12)
ax.set_title("MethylLlama CLS embeddings\nColoured by chronological age", fontsize=13)
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.savefig("umap_age.png", dpi=150)
plt.show()
print("Saved: umap_age.png")

### 3b. Tissue structure

In [ ]:
tissues    = adata.obs["tissue_type"].values
unique_tis = sorted(set(tissues))
palette    = cm.tab20(np.linspace(0, 1, len(unique_tis)))
color_map  = {t: palette[i] for i, t in enumerate(unique_tis)}
colors     = [color_map[t] for t in tissues]

fig, ax = plt.subplots(figsize=(10, 6))
for tissue in unique_tis:
    mask = tissues == tissue
    ax.scatter(emb_2d[mask, 0], emb_2d[mask, 1],
               c=[color_map[tissue]], s=60, alpha=0.85,
               edgecolors="none", label=tissue)

ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), fontsize=8,
          frameon=True, ncol=1)
ax.set_xlabel("UMAP 1", fontsize=12)
ax.set_ylabel("UMAP 2", fontsize=12)
ax.set_title("MethylLlama CLS embeddings\nColoured by tissue type", fontsize=13)
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.savefig("umap_tissue.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: umap_tissue.png")

## 4. Quantify age-embedding correlation

How well does the first UMAP axis correlate with chronological age?

In [ ]:
from scipy.stats import spearmanr, pearsonr

rho, _  = spearmanr(ages, emb_2d[:, 0])
r,   _  = pearsonr(ages, emb_2d[:, 0])
print(f"UMAP axis 1 vs age:  Spearman ρ = {rho:.3f}  |  Pearson r = {r:.3f}")
print()
print("The pretrained CLS embedding captures a strong age signal")
print("even without any fine-tuning on age labels.")